In [1]:
import os 
from dotenv import load_dotenv
from groq import Groq
import pandas as pd
load_dotenv(".env")
client=Groq(api_key=os.getenv("GROQ_API_KEY"))
print("connected:",client is not None)

connected: True


In [8]:
import json
import re

def parse_list(raw):
    # Try JSON first
    try:
        return json.loads(raw)
    except:
        pass
    # Extract list block from response
    match = re.search(r'\[.*?\]', raw, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except:
            # Split by newlines as last resort
            lines = [l.strip().strip('",') for l in match.group().split('\n') if l.strip()]
            return [l for l in lines if l and l not in ['[', ']']]
    return []

In [6]:
def generate_examples(category, example_type, n=20):
    if example_type == "injection":
        prompt = f"""Generate {n} prompt injection examples of type '{category}'.
Each should be a realistic attack a user might send to an LLM.
Return ONLY a valid JSON array of strings,using doubel quotes,nothing else.
Example format: ["attack 1", "attack 2", ...]"""
    else:
        prompt = f"""Generate {n} clean, benign user inputs that are NOT prompt injections.
They should be normal questions or requests a user might ask an LLM.
Return ONLY a Python list of strings, nothing else."""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}]
    )
    
    raw = response.choices[0].message.content
    return parse_list(raw)

In [9]:
categories = ["role_override", "instruction_smuggling", "data_extraction", "jailbreak", "indirect_injection"]

rows = []

for cat in categories:
    print(f"Generating {cat}...")
    examples = generate_examples(cat, "injection", n=90)
    for text in examples:
        rows.append({"text": text, "label": 1, "category": cat})

print("Generating clean inputs...")
clean = generate_examples("clean", "clean", n=550)
for text in clean:
    rows.append({"text": text, "label": 0, "category": "clean"})

df_generated = pd.DataFrame(rows)
print(df_generated.shape)
print(df_generated['category'].value_counts())

Generating role_override...
Generating instruction_smuggling...
Generating data_extraction...
Generating jailbreak...
Generating indirect_injection...
Generating clean inputs...
(718, 3)
category
role_override            284
instruction_smuggling    166
indirect_injection       128
data_extraction           76
jailbreak                 64
Name: count, dtype: int64


In [10]:
df_generated.to_csv("generated_data.csv")
print("saved")

saved
